# KLWX four-hour case timeline

This stays with the June 5, 2024 Washington, DC-area case used by the tracker. It samples one real KLWX Level II scan every 30 minutes for the four hours leading to the selected 23:23 UTC scan. The goal is case context: see what developed, weakened, and moved through the region before evaluating the short-lead McLean forecast.

This is **not** a claim that one individual storm cell remains trackable for ten hours. The operational cluster tracker and the three-radar McLean mosaic remain in notebook 03.

In [ ]:
%pip install -e "../.[nexrad]"

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import re
import sys
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))
from storm_motion.impact import local_xy_from_lat_lon
from storm_motion.nexrad import load_lowest_reflectivity_sweep

bucket = 'unidata-nexrad-level2'
prefix = '2024/06/05/KLWX/'
data_dir = repo / 'data' / 'klwx_20240605_timeline'
data_dir.mkdir(parents=True, exist_ok=True)
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

In [ ]:
# Select the nearest actual volume to each half-hour target during the four-hour lead-in.
case_end = datetime(2024, 6, 5, 23, 23, tzinfo=timezone.utc)
case_start = case_end - timedelta(hours=4)
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
pattern = re.compile(r'KLWX(\d{8})_(\d{6})_V06$')
available = []
for item in response.get('Contents', []):
    match = pattern.search(item['Key'])
    if match:
        observed_at = datetime.strptime(''.join(match.groups()), '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)
        if case_start - timedelta(minutes=10) <= observed_at <= case_end + timedelta(minutes=10):
            available.append((observed_at, item['Key']))
if not available:
    raise RuntimeError('No KLWX files found for the requested case window.')

targets = [case_start + timedelta(minutes=30 * index) for index in range(9)]
selected = []
used_keys = set()
for target in targets:
    observed_at, key = min(available, key=lambda item: abs(item[0] - target))
    if key not in used_keys:
        selected.append((observed_at, key))
        used_keys.add(key)

timeline_paths = []
for observed_at, key in selected:
    path = data_dir / Path(key).name
    if not path.exists():
        s3.download_file(bucket, key, str(path))
    timeline_paths.append(path)
print(f'Loaded {len(timeline_paths)} real KLWX volumes from {selected[0][0]:%H:%M} to {selected[-1][0]:%H:%M} UTC.')

In [ ]:
office_latitude_deg, office_longitude_deg = 38.922403, -77.232125
timeline_summary = []
fig, axes = plt.subplots(3, 3, figsize=(14, 13), sharex=True, sharey=True)
for ax, path in zip(axes.flat, timeline_paths):
    # Load, summarize, and render one volume at a time to stay within 4 GB RAM.
    frame = load_lowest_reflectivity_sweep(path)
    office_x_m, office_y_m = local_xy_from_lat_lon(
        office_latitude_deg, office_longitude_deg,
        frame.radar_latitude_deg, frame.radar_longitude_deg,
    )
    distance_to_office_m = np.hypot(frame.x_m - office_x_m, frame.y_m - office_y_m)
    local = (distance_to_office_m <= 25_000) & np.isfinite(frame.reflectivity_dbz)
    strong = local & (frame.reflectivity_dbz >= 35)
    timeline_summary.append((
        frame.observed_at,
        float(np.nanmax(np.where(local, frame.reflectivity_dbz, np.nan))),
        float(np.mean(strong[local])) if np.any(local) else np.nan,
    ))
    image = ax.pcolormesh(frame.x_m / 1_000, frame.y_m / 1_000, frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
    ax.scatter(office_x_m / 1_000, office_y_m / 1_000, marker='*', s=90, c='#ff4d4d', edgecolors='white', linewidths=.7, zorder=5)
    ax.set_title(f'{frame.observed_at:%H:%M UTC}', fontsize=9)
    ax.set(xlim=(-100, 120), ylim=(-100, 130), aspect='equal')
fig.supxlabel('km east of KLWX'); fig.supylabel('km north of KLWX')
fig.suptitle('KLWX: four-hour real-radar context leading to the McLean forecast case', fontsize=15, weight='bold')
fig.colorbar(image, ax=axes.ravel().tolist(), label='reflectivity (dBZ)', shrink=.75)
plt.show(); plt.close(fig)

In [ ]:
# Compact quantitative context: strong-reflectivity coverage within 25 km of McLean at every sampled scan.
print('MCLEAN LOCAL RADAR CONTEXT (KLWX)')
for observed_at, max_dbz, strong_fraction in timeline_summary:
    print(f'{observed_at:%H:%M UTC}: max {max_dbz:4.1f} dBZ; {strong_fraction:5.1%} of sampled gates within 25 km at or above 35 dBZ')

print('Open notebook 03 next for the detailed three-radar mosaic, Kalman motion estimate, and McLean impact outlook at the selected end time.')